# Week 1: train ONE ensemble member (60 classes, n_fft 64)

Each teammate runs this once with **their own seed**. Do not change the seed or any setting after you start.

| Seed | Who |
|---|---|
| 1, 2 | Eavan |
| 3 | Chua |
| 4 | Jessy |
| 5 | Eileen |

**Before you start:** Runtime → Change runtime type → **T4 GPU**.

**If Colab disconnects:** just run every cell again from the top with the same seed. Training resumes from the last finished epoch (`last.pt` is saved to Drive after every epoch).

## 1. Settings: change only `SEED` (and `FOI26` if your Drive path is different)

In [ ]:
SEED = 1          # <-- your seed: Eavan 1 and 2, Chua 3, Jessy 4, Eileen 5
BRANCH = 'eavan-merge-shards'

# The shared FOI26 folder in Drive. If it was shared WITH you, open it in Drive,
# right-click -> Organise -> Add shortcut -> My Drive, so it appears here.
FOI26 = '/content/drive/MyDrive/sedic/FOI26'

DATA_ON_DRIVE = f'{FOI26}/combined_data'           # train/ val/ test/ + combined.json
# One folder per run inside results/: training date + what is being tried + seed.
# Type the date by hand (not today's date from code): if Colab disconnects and you
# rerun tomorrow, the SAME name is what lets training resume instead of restarting.
RUN_NAME = '2026-11-04_nfft64_ep15'   # <-- date_feature; change it only for a NEW experiment
EPOCHS = 15                           # 15 = first check (~1-1.5 h); 30 = the real ensemble run
CKPT_DIR      = f'{FOI26}/results/{RUN_NAME}_seed{SEED}'  # last.pt, best.pt, history.json, scorecard.json
# Only for the real ensemble run: then set it to f'{FOI26}/members/seed{SEED}.pt'.
# Left empty for a first check, so a trial model never lands in members/.
MEMBER_OUT    = ''

## 2. GPU check

In [ ]:
import torch
print('CUDA:', torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only: change the runtime to T4 GPU first')

## 3. Mount Drive and get the code

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

%cd /content
!rm -rf sedicAI_NEXA
!git clone -q -b {BRANCH} https://github.com/eavan127/sedicAI_NEXA.git
%cd /content/sedicAI_NEXA
!git log -1 --format='commit %H  %s'
!pip install -q pyyaml

## 4. Copy the data to the Colab disk

Reading straight from Drive during training is very slow, so copy it once (~2.4 GB, a few minutes).
The check below stops here if the copy is incomplete.

In [ ]:
import json, os, numpy as np
!mkdir -p /content/combined
!cp -r "{DATA_ON_DRIVE}/train" "{DATA_ON_DRIVE}/val" "{DATA_ON_DRIVE}/test" /content/combined/
!cp "{DATA_ON_DRIVE}/combined.json" /content/combined/ || echo 'combined.json missing on Drive: upload it from data/combined/'

for split, n in (('train', 960_000), ('val', 120_000), ('test', 120_000)):
    X = np.load(f'/content/combined/{split}/X.npy', mmap_mode='r')
    y = np.load(f'/content/combined/{split}/y.npy', mmap_mode='r')
    print(split, X.shape, X.dtype, y.shape, y.dtype)
    assert X.shape == (n, 2, 512) and X.dtype in (np.float16, np.float32) and y.shape == (n, 60), f'{split} looks wrong: re-upload it'
if os.path.exists('/content/combined/combined.json'):
    print('combined version:', json.load(open('/content/combined/combined.json'))['combined_version'], '(expect de26671a5d48)')

## 5. Quick check (about 1 minute)

Trains on 2,000 windows for 1 epoch into a throwaway folder, just to prove everything runs. **If this fails, stop and send the error to Eavan.**

In [ ]:
!python scripts/train_week1.py --seed {SEED} --limit 2000 --epochs 1 --batch-size 256 --ckpt-dir /content/try --fresh

## 6. Train (about 1-1.5 h for 15 epochs, 2-3 h for 30, on a T4)

Prints one line per epoch with the time left. Every epoch is saved to `CKPT_DIR` (`results/<RUN_NAME>_seed<k>/`) on Drive.
When it finishes it scores the test split (`scorecard.json`) and, if `MEMBER_OUT` is set, copies the best model there.

**Write in your run log:** RUN_NAME, seed, the commit printed in step 3, the combined version printed in step 4.


In [ ]:
member = f'--member-out "{MEMBER_OUT}"' if MEMBER_OUT else ''
!python scripts/train_week1.py --seed {SEED} --epochs {EPOCHS} --batch-size 256 --ckpt-dir "{CKPT_DIR}" {member} --eval-test


## 7. Results

In [ ]:
h = json.load(open(f'{CKPT_DIR}/history.json'))
print('best epoch:', h['best_epoch'])
for e in h['epochs']:
    print(f"{e['epoch']:>2}  val_loss {e['val_loss']:.4f}  top1 {e['val_top1']:.3f}  recall@0.5 {e['val_macro_recall_05']:.3f}  {e['seconds']:.0f}s")
if os.path.exists(f'{CKPT_DIR}/scorecard.json'):
    t = json.load(open(f'{CKPT_DIR}/scorecard.json'))['test']
    print(f"
TEST  top1 {t['top1']:.3f}  macro recall@0.5 {t['macro_recall_05']:.3f}")
    worst = sorted(t['per_class'].items(), key=lambda kv: kv[1]['top1_standalone'])[:10]
    print('10 weakest classes (top-1):', [(c, round(v['top1_standalone'], 2)) for c, v in worst])